# FairMALLM-GAN v3 — CTGAN baseline (RQ3)

Accelerator GPU (any), Internet on, input **oulad-processed**. Trains CTGAN on the flattened weekly panel and evaluates it with the same GRU protocol.

In [ ]:
!pip install -q ctgan


In [ ]:
import glob, shutil
for f in ["common.py", "fair_mallm_gan_v3.py", "evaluate_v3.py", "baseline_ctgan.py", "prepare_cohort.py"]:
    src = glob.glob(f"/kaggle/input/**/{f}", recursive=True)
    assert src, f"{f} not found - attach the dataset 'fairmallm-v3-code' (Add Input)"
    shutil.copy(src[0], f"/kaggle/working/{f}")
print("code copied")


In [ ]:
import glob, os, sys, json, shutil, torch
sys.path.insert(0, "/kaggle/working"); os.chdir("/kaggle/working")
hits = glob.glob("/kaggle/input/**/oulad_processed/X_seq.npy", recursive=True) or \
       glob.glob("/kaggle/input/**/X_seq.npy", recursive=True)
assert hits, "Attach the dataset 'oulad-processed' (Add Input) first."
DATA_DIR = os.path.dirname(hits[0])
print("DATA_DIR =", DATA_DIR)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
os.environ["DATA_DIR"] = DATA_DIR


In [ ]:
!python baseline_ctgan.py --data_dir "$DATA_DIR" --out_dir /kaggle/working/run_ctgan --n_synthetic 2000 --epochs 300
from evaluate_v3 import run as run_eval
res = run_eval(DATA_DIR, "/kaggle/working/eval_ctgan", {"ctgan": "/kaggle/working/run_ctgan"},
               cohort="active", seeds=(0, 1, 2, 3, 4))
print(open("/kaggle/working/eval_ctgan/report.md").read())
# sensitivity: TRTR on ALL students (incl. those who already withdrew before the cutoff)
res_all = run_eval(DATA_DIR, "/kaggle/working/eval_trtr_all_students", {}, cohort="all", seeds=(0, 1, 2))
print(open("/kaggle/working/eval_trtr_all_students/report.md").read())
